# Optiver Realized Volatility — Exploration

In [1]:
import pandas as pd

train = pd.read_csv("data/train.csv")
print(train.shape)
print(train.columns)
train.head()

(428932, 3)
Index(['stock_id', 'time_id', 'target'], dtype='str')


,stock_id,time_id,target
0,0,5,0.004136
1,0,11,0.001445
2,0,16,0.002168
3,0,31,0.002195
4,0,62,0.001747


In [2]:
# book_train.parquet is partitioned by stock_id; only loading stock 0 for now.
book_stock_0 = pd.read_parquet("data/book_train.parquet/stock_id=0")
print(book_stock_0.shape)
print(book_stock_0.columns)
book_stock_0.head()

(917553, 10)
Index(['time_id', 'seconds_in_bucket', 'bid_price1', 'ask_price1',
       'bid_price2', 'ask_price2', 'bid_size1', 'ask_size1', 'bid_size2',
       'ask_size2'],
      dtype='str')


,time_id,seconds_in_bucket,bid_price1,ask_price1,bid_price2,ask_price2,bid_size1,ask_size1,bid_size2,ask_size2
0,5,0,1.001422,1.002301,1.00137,1.002353,3,226,2,100
1,5,1,1.001422,1.002301,1.00137,1.002353,3,100,2,100
2,5,5,1.001422,1.002301,1.00137,1.002405,3,100,2,100
3,5,6,1.001422,1.002301,1.00137,1.002405,3,126,2,100
4,5,7,1.001422,1.002301,1.00137,1.002405,3,126,2,100


In [3]:
one_window=book_stock_0[book_stock_0["time_id"]==5]
print(one_window.head())

   time_id  seconds_in_bucket  bid_price1  ask_price1  bid_price2  ask_price2  \
0        5                  0    1.001422    1.002301     1.00137    1.002353   
1        5                  1    1.001422    1.002301     1.00137    1.002353   
2        5                  5    1.001422    1.002301     1.00137    1.002405   
3        5                  6    1.001422    1.002301     1.00137    1.002405   
4        5                  7    1.001422    1.002301     1.00137    1.002405   

   bid_size1  ask_size1  bid_size2  ask_size2  
0          3        226          2        100  
1          3        100          2        100  
2          3        100          2        100  
3          3        126          2        100  
4          3        126          2        100  


In [4]:
# numerator: cross-weight each side's price by the OTHER side's size, then add
one_window["wap"]=(
     (one_window["bid_price1"] * one_window["ask_size1"]  # best buy price weighted by sell-side size
     + one_window["ask_price1"] * one_window["bid_size1"])  # + best sell price weighted by buy-side size
    / (one_window["bid_size1"] + one_window["ask_size1"])  # divide by total size at the best bid+ask
)
import numpy as np
#return is how much the price moved from one moment to the next 
#we use log because it is additive and symmetric so a move up then an equal move down will cancel to 0
one_window["log_return"]=np.log(one_window["wap"]/one_window["wap"].shift(1))
print(one_window[["seconds_in_bucket", "wap", "log_return"]].head())

   seconds_in_bucket       wap  log_return
0                  0  1.001434         NaN
1                  1  1.001448    0.000014
2                  5  1.001448    0.000000
3                  6  1.001443   -0.000005
4                  7  1.001443    0.000000


In [5]:
#realized volatility
realized_vol=np.sqrt((one_window["log_return"]**2).sum())
print("realized vol (computed):", realized_vol)
#get row with both stock_id=0 and time_id=5 for target
target_row=train[(train["stock_id"]==0) & (train["time_id"]==5)]
target_value=target_row["target"].iloc[0]
print("target from train.csv: ", target_value )

realized vol (computed): 0.004499364172786568
target from train.csv:  0.004135767


In [6]:
#WAP for every row, not just one window
book_stock_0["wap"]=(book_stock_0["bid_price1"]*book_stock_0["ask_size1"]+
                     book_stock_0["ask_price1"] * book_stock_0["bid_size1"]) / (book_stock_0["bid_size1"]+ book_stock_0["ask_size1"])
book_stock_0[["bid_price1", "ask_price1", "wap"]].head()

#log return per row, shifted within each time_id window only
#groupby prevents comparing the last row of one window to the first row of the next 
book_stock_0["log_return"]=np.log(book_stock_0["wap"]/ book_stock_0.groupby("time_id")["wap"].shift(1))

#realized vol=sqrft(sum of squared log returns), one value per time_id window 
realized_vol_per_window=(book_stock_0.groupby("time_id")["log_return"]
                         .apply(lambda x: np.sqrt((x**2).sum()))
                         .rename("realized_vol"))
print(realized_vol_per_window.head())

#sanity check 
stock_0_targets=train[train["stock_id"]==0].set_index("time_id")["target"]
comparison=pd.concat([realized_vol_per_window, stock_0_targets], axis=1)
print(comparison.head(10)) # close in magnitude and move together 


time_id
5     0.004499
11    0.001204
16    0.002369
31    0.002574
62    0.001894
Name: realized_vol, dtype: float64
         realized_vol    target
time_id                        
5            0.004499  0.004136
11           0.001204  0.001445
16           0.002369  0.002168
31           0.002574  0.002195
62           0.001894  0.001747
72           0.007902  0.004912
97           0.010034  0.009388
103          0.005331  0.004120
109          0.001797  0.002182
123          0.003273  0.002669


Volatility clustering = naive baseline 

In [7]:
def rmspe(y_true, y_pred):
    return np.sqrt(np.mean(((y_true-y_pred)/y_true)**2))

score_stock_0=rmspe(comparison["target"], comparison["realized_vol"])
print("RSMPE for stock 0 naive baseline: ", score_stock_0)

RSMPE for stock 0 naive baseline:  0.39322701284497674


In [8]:
def compute_realized_vol_for_stock(stock_id):
    book = pd.read_parquet(f"data/book_train.parquet/stock_id={stock_id}")
    book["wap"] = (
        (book["bid_price1"] * book["ask_size1"] + book["ask_price1"] * book["bid_size1"])
        / (book["bid_size1"] + book["ask_size1"])
    )
    book["log_return"] = np.log(book["wap"] / book.groupby("time_id")["wap"].shift(1))
    realized_vol = (
        book.groupby("time_id")["log_return"]
        .apply(lambda x: np.sqrt((x**2).sum()))
        .rename("realized_vol")
        .reset_index()
    )
    realized_vol["stock_id"] = stock_id
    return realized_vol

In [9]:
subset_stock_ids = [0, 1, 2, 3]
all_predictions = pd.concat(
    [compute_realized_vol_for_stock(s) for s in subset_stock_ids],
    ignore_index=True,
)
print(all_predictions.shape)
all_predictions.head()

(15320, 3)


,time_id,realized_vol,stock_id
0,5,0.004499,0
1,11,0.001204,0
2,16,0.002369,0
3,31,0.002574,0
4,62,0.001894,0


In [10]:
merged = all_predictions.merge(train, on=["stock_id", "time_id"])
naive_baseline_rmspe = rmspe(merged["target"], merged["realized_vol"])
print("Naive baseline RMSPE across stocks", subset_stock_ids, ":", naive_baseline_rmspe)

Naive baseline RMSPE across stocks [0, 1, 2, 3] : 0.33583164444718683


The naive baseline predictions are off by aprox. 34% on average.  

Refactor compute_realized_vol_for_stock to return a whole feature table instead of a column 

Added a "recent volatility" feature . The end of a window is closer in time to what we are expecting for the next window, so recent turbulence may matter more. 

In [11]:
def compute_features_for_stock(stock_id):
    """Read one stock's order book and return ONE row of features per time_id window."""
    book = pd.read_parquet(f"data/book_train.parquet/stock_id={stock_id}")

    book["wap"] = (
        (book["bid_price1"] * book["ask_size1"] + book["ask_price1"] * book["bid_size1"])
        / (book["bid_size1"] + book["ask_size1"])
    )

    book["log_return"] = np.log(book["wap"] / book.groupby("time_id")["wap"].shift(1))

    book["spread"] = book["ask_price1"] - book["bid_price1"]
    book["size_imbalance"] = book["bid_size1"] / (book["bid_size1"] + book["ask_size1"])

    # window (seconds_in_bucket >= 300) the part closest to the next window.
    last_300 = book[book["seconds_in_bucket"] >= 300]
    realized_vol_last_300 = (
        last_300.groupby("time_id")["log_return"]
        .apply(lambda x: np.sqrt((x ** 2).sum()))
    )

    
    grouped = book.groupby("time_id")
    features = pd.DataFrame({
        "realized_vol": grouped["log_return"].apply(lambda x: np.sqrt((x**2).sum())),
        "realized_vol_last_300": realized_vol_last_300,   
        "spread_mean": grouped["spread"].mean(),
        "size_imbalance_mean": grouped["size_imbalance"].mean(),
    }).reset_index()

    # fill any window missing a last-300s value with its full-window vol
    features["realized_vol_last_300"] = features["realized_vol_last_300"].fillna(
        features["realized_vol"]
    )

    features["stock_id"] = stock_id
    return features

In [12]:
subset_stock_ids = [0, 1, 2, 3]
features_table = pd.concat(
    [compute_features_for_stock(s) for s in subset_stock_ids],
    ignore_index=True,
)
print(features_table.shape)
features_table.head()

(15320, 6)


,time_id,realized_vol,realized_vol_last_300,spread_mean,size_imbalance_mean,stock_id
0,5,0.004499,0.002953,0.000855,0.496821,0
1,11,0.001204,0.000981,0.000394,0.583582,0
2,16,0.002369,0.001295,0.000725,0.447253,0
3,31,0.002574,0.001776,0.000859,0.528189,0
4,62,0.001894,0.001520,0.000397,0.544335,0


In [13]:
features_table = features_table.merge(train, on=["stock_id", "time_id"])
print(features_table.shape)
features_table.head()

(15320, 7)


,time_id,realized_vol,realized_vol_last_300,spread_mean,size_imbalance_mean,stock_id,target
0,5,0.004499,0.002953,0.000855,0.496821,0,0.004136
1,11,0.001204,0.000981,0.000394,0.583582,0,0.001445
2,16,0.002369,0.001295,0.000725,0.447253,0,0.002168
3,31,0.002574,0.001776,0.000859,0.528189,0,0.002195
4,62,0.001894,0.001520,0.000397,0.544335,0,0.001747


Which features actually move with the target? 
.corr() gives a number in [-1, 1]: near +1 = strong positive relationship,
near 0 = little linear relationship. 

In [14]:
feature_cols = ["realized_vol", "realized_vol_last_300", "spread_mean", "size_imbalance_mean"]
print(features_table[feature_cols + ["target"]].corr()["target"].sort_values(ascending=False))

target                   1.000000
realized_vol_last_300    0.884096
realized_vol             0.873876
spread_mean              0.734535
size_imbalance_mean     -0.044672
Name: target, dtype: float64


In [15]:
import lightgbm as lgb
from sklearn.model_selection import GroupKFold


features_table["imbalance_intensity"] = (features_table["size_imbalance_mean"] - 0.5).abs()


feature_cols = [
    "realized_vol",
    "realized_vol_last_300",
    "spread_mean",
    "size_imbalance_mean",
    "imbalance_intensity",
]
X = features_table[feature_cols]
y = features_table["target"]
groups = features_table["time_id"]   # keep all rows of one market moment in the same fold

# 5-fold GROUP cross-validation
gkf = GroupKFold(n_splits=5)
# "out-of-fold" predictions: each row is predicted by a model that didn't train on it
oof_pred = np.zeros(len(features_table))

for fold, (train_idx, val_idx) in enumerate(gkf.split(X, y, groups)):
    model = lgb.LGBMRegressor(
        n_estimators=500,      # number of trees
        learning_rate=0.05,    # how much each tree corrects the previous ones
        num_leaves=31,         # tree complexity
        random_state=42,       # reproducibility
        verbose=-1,            # silence per-tree logging
    )
    # sample_weight = 1/y^2 makes training optimize PERCENTAGE error (matches RMSPE)
    weights = 1.0 / np.square(y.iloc[train_idx])
    model.fit(X.iloc[train_idx], y.iloc[train_idx], sample_weight=weights)
    oof_pred[val_idx] = model.predict(X.iloc[val_idx])

# score every window using the prediction made while it was held out
model_rmspe = rmspe(y.values, oof_pred)
print("Naive baseline RMSPE:", 0.336)
print("LightGBM CV RMSPE:", round(model_rmspe, 5))
print("Improvement:", f"{(1 - model_rmspe/0.336)*100:.1f}% lower error")

Naive baseline RMSPE: 0.336
LightGBM CV RMSPE: 0.25039
Improvement: 25.5% lower error


Naive baseline RMSPE: 0.336
LightGBM CV RMSPE: 0.25039
Improvement: 25.5% lower error

Model selection;  justifying the hyperparameters

The first model used hand-picked values n_estimators=500, learning_rate=0.05,
num_leaves=31. They are sensible defaults, but not derived from the data. Here we justify
them:

1. RandomizedSearchCV tries 12 random `(learning_rate, num_leaves)` combinations and
   keeps the one with the best cross-validated RMSPE.
2. Early stopping then lets a held-out validation slice choose the number of trees,
   instead of hardcoding it, training stops once validation RMSPE stops improving.

In [16]:

import scipy.stats
from sklearn.model_selection import RandomizedSearchCV, GroupShuffleSplit
from sklearn.metrics import make_scorer

# RMSPE is an error (lower = better); make_scorer negates it so sklearn can "maximize"
rmspe_scorer = make_scorer(rmspe, greater_is_better=False)

# Randomized search
param_dist = {
    "learning_rate": scipy.stats.uniform(0.01, 0.09),  # uniform(loc, scale) -> range [0.01, 0.10]
    "num_leaves": [15, 31, 63],
}

search = RandomizedSearchCV(
    lgb.LGBMRegressor(n_estimators=300, random_state=42, verbose=-1, n_jobs=1),
    param_dist,
    n_iter=12,          # try 12 random combinations (x5 folds = 60 quick fits)
    cv=gkf,             # same GroupKFold (split by time_id) used everywhere
    scoring=rmspe_scorer,
    random_state=42,
    n_jobs=1,           # single-threaded: LightGBM already parallelizes internally
)
search.fit(X, y, groups=groups)
best_params = search.best_params_   
print("chosen learning_rate:", round(best_params["learning_rate"], 4))
print("chosen num_leaves   :", best_params["num_leaves"])

# Early stopping
def rmspe_eval(y_true, y_pred):
    return "rmspe", rmspe(y_true, y_pred), False   # (name, value, is_higher_better)

trees_per_fold = []
for train_idx, val_idx in gkf.split(X, y, groups):
    X_tr, y_tr, g_tr = X.iloc[train_idx], y.iloc[train_idx], groups.iloc[train_idx]
    # hold out 20% of the training groups purely to decide when to stop adding trees;
    # the outer fold stays untouched, so it can still give an honest score later.
    inner = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
    fit_idx, stop_idx = next(inner.split(X_tr, y_tr, g_tr))
    tuner = lgb.LGBMRegressor(
        n_estimators=2000,                            # upper limit; early stopping picks the real number
        learning_rate=best_params["learning_rate"],
        num_leaves=best_params["num_leaves"],
        random_state=42, verbose=-1, n_jobs=1,
    )
    tuner.fit(
        X_tr.iloc[fit_idx], y_tr.iloc[fit_idx],
        sample_weight=1.0 / np.square(y_tr.iloc[fit_idx]),
        eval_set=[(X_tr.iloc[stop_idx], y_tr.iloc[stop_idx])],
        eval_metric=rmspe_eval,
        callbacks=[lgb.early_stopping(50, verbose=False)],   # stop after 50 no-improve rounds
    )
    trees_per_fold.append(int(tuner.best_iteration_))

best_n_estimators = int(np.median(trees_per_fold))   
print("trees chosen per fold:", trees_per_fold)
print("chosen n_estimators (median):", best_n_estimators)

chosen learning_rate: 0.0437
chosen num_leaves   : 15
trees chosen per fold: [133, 126, 157, 96, 107]
chosen n_estimators (median): 126


In [17]:
# Final model: reads the computed hyperparameters above 
# best_params and best_n_estimators come from the compute cell right above.
oof_pred_tuned = np.zeros(len(features_table))   # out-of-fold predictions

for train_idx, val_idx in gkf.split(X, y, groups):
    final_model = lgb.LGBMRegressor(
        n_estimators=best_n_estimators,               # chosen by early stopping
        learning_rate=best_params["learning_rate"],   #chosen by the randomized search
        num_leaves=best_params["num_leaves"],         #chosen by the randomized search
        random_state=42, verbose=-1, n_jobs=1,
    )
    final_model.fit(
        X.iloc[train_idx], y.iloc[train_idx],
        sample_weight=1.0 / np.square(y.iloc[train_idx]),   # optimize percentage error (RMSPE)
    )
    oof_pred_tuned[val_idx] = final_model.predict(X.iloc[val_idx])

tuned_rmspe = rmspe(y.values, oof_pred_tuned)
print("Naive baseline RMSPE :", 0.336)
print("Tuned LightGBM RMSPE :", round(tuned_rmspe, 5))
print("Improvement          :", f"{(1 - tuned_rmspe / 0.336) * 100:.1f}% lower error")

Naive baseline RMSPE : 0.336
Tuned LightGBM RMSPE : 0.24466
Improvement          : 27.2% lower error


Naive baseline RMSPE : 0.336
Tuned LightGBM RMSPE : 0.24466
Improvement          : 27.2% lower error, 2% better than the model with the initial parameters based on general guidelines